# 第 79 课｜推理服务事故演练：SLO 双窗口、灰度回滚与恢复验证

上一课把 Trace、结构化日志与 SLO 指标连到**真实增量 KV 推理服务**。这一课回答更关键的问题：看到事故信号后，**凭什么判断应该回滚？回滚是否真正恢复？**

完整链路：两版本模型 → 请求固定路由 → 真实增量 KV 计算 → 终态 SLI → 双窗口错误预算消耗率 → 灰度归因 → 回滚锁存 → 物理页回收 → 新流量恢复验证 → 审计。

**只用已有 PyTorch 环境，默认离线 CPU；Notebook 和同名 `.py` 各自独立。**

**严格的边界：**模型是随机初始化的小型因果 Transformer（Q/V LoRA 两版本），并不具备语言质量基准能力；真实 K/V 存储和注意力计算，但服务、故障和持续时间使用确定性**逻辑 tick**，不是 HTTP、OpenTelemetry SDK、GPU 毫秒延迟或生产规模演练。脚本不会访问外部服务、发布真实版本或触发真实回滚。

## 0. 推荐运行方式与验收顺序

从**全新内核**依次运行。完整实验默认执行，生成新的时间戳目录。你也可以先把 `RUN_FULL_DRILLS=False`，只看数学、预检和练习。

本课最后包含 162 项固定回归检查与三道默认关闭的独立练习。所有测试通过，不等于已经做过真实网络压测或云服务事故演练。

脚本等价命令：

```bash
python lesson_79_serving_incident_slo_rollback.py --checks-only
python lesson_79_serving_incident_slo_rollback.py --math-only
python lesson_79_serving_incident_slo_rollback.py
python lesson_79_serving_incident_slo_rollback.py --reload-run lesson_79_outputs/实际目录
```

先记住本课结论：**灰度特有故障可以有证据地回滚；两个版本同时变慢不应被草率归咎于新版本；回滚动作发生不代表事故已经恢复。**

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第79课：推理服务故障演练、双窗口SLO和可审计自动回滚。

完全离线、CPU、小型随机Transformer与真实增量KV页池。
注入的慢请求及错误是确定性逻辑tick，不是GPU毫秒、真实HTTP或生产事故数据。
不自动联网、不下载模型、不运行真实服务、不更改任何外部发布状态。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import json
import math
import platform
import random
import re
import secrets
import tempfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("本课需要已有的PyTorch环境；不会自动安装软件或下载权重。") from exc

VERSION = "lesson79-incident-drills-v1"
SOURCES = {
 "sre_alerting":"https://sre.google/workbook/alerting-on-slos/",
 "sre_incidents":"https://sre.google/sre-book/managing-incidents/",
 "aws_rollback":"https://docs.aws.amazon.com/codedeploy/latest/userguide/deployments-rollback-and-redeploy.html",
 "otel_traces":"https://opentelemetry.io/docs/concepts/context-propagation/",
 "pytorch_inference":"https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html",
 "reproducibility":"https://docs.pytorch.org/docs/stable/notes/randomness.html",
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
print('Python:',platform.python_version())
print('PyTorch:',torch.__version__,'CUDA可用:',torch.cuda.is_available())
print('本课的延迟单位：logic ticks；这里没有计量真实GPU/网络毫秒。')

Python: 3.13.5
PyTorch: 2.10.0+cpu CUDA可用: False
本课的延迟单位：logic ticks；这里没有计量真实GPU/网络毫秒。


## 1. 先固定“模型身份”：两个 LoRA 版本不能在同一 KV 序列里互换

我们重建一份随机基座，给 stable 和 canary 分别注入 Q/V LoRA。两者拥有**完全相同基座**，但 adapter 张量不同。两份服务模型均冻结。

真实服务必须把 tokenizer、Chat Template、基座、adapter、精度和缓存范围视为模型身份的一部分。本课只实现固定教学 tokenizer 与两版本 LoRA，不声称可直接加载真实 Qwen。

之后同一个请求在准入时就会绑定版本。即便路由权重从 50% 降到 0%，**已经准入的旧请求也不能在生成中途换模型**，否则 KV 可能来自错误参数。

In [3]:
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')

class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)

class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))

class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)

def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy

def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]

def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}

def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))

def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base

def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [4]:
def build_servable_models(seed=7901):
    require(type(seed) is int and seed>=0,'seed非法')
    seed_all(seed)
    tok=ByteMergeChatTokenizer()
    cfg=ToyConfig(tok.vocab_size,dim=16,heads=2,layers=1,ff_dim=32,max_length=96)
    base=TinyChatLM(cfg).eval()
    stable=inject_lora(base,rank=4,alpha=8.0)
    canary=inject_lora(base,rank=4,alpha=8.0)
    gen=torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for name,p in adapters(canary):
            if name.endswith('.B'):
                p.copy_(torch.randn(p.shape,generator=gen)*.1)
    models={'stable':stable.eval().requires_grad_(False),'canary':canary.eval().requires_grad_(False)}
    require(base_hash(models['stable'])==base_hash(models['canary']),'不是同一基座')
    require(state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()),'两个版本实际权重相同')
    return tok,models

In [5]:
tok, models = build_servable_models(7901)
print('词表大小:',tok.vocab_size)
print('基座参数:',sum(v.numel() for v in base_state(models['stable']).values()))
print('每版本LoRA参数:',sum(p.numel() for _,p in adapters(models['stable'])))
print('基座身份相同:',base_hash(models['stable'])==base_hash(models['canary']))
print('完整版本权重不同:',state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()))
print('参考/灰度服务参数均被冻结:',all(not p.requires_grad for m in models.values() for p in m.parameters()))

词表大小: 281
基座参数: 12720
每版本LoRA参数: 256
基座身份相同: True
完整版本权重不同: True
参考/灰度服务参数均被冻结: True


## 2. 必须有真实 KV 计算，不能用随机数字伪造“缓存已运行”

这一单元实现 CPU 物理 KV 张量，形状是 `[blocks,layers,K_or_V,slots,heads,head_dim]`，并为每个会话维护块表、引用数、提交与回滚。

缓存 decode 每增加一个 token，都实际计算 Q/K/V，把 K/V 写入页池，再读取此前所有 K/V 做注意力。这不是 vLLM 的 PagedAttention 高性能内核，只是可核对的教学页池。

关键性质：**相同版本、相同输入时，增量 logits 必须与完整上下文的因果前向一致。** 另外，末尾 KV 和整个会话释放后，页池需回到零占用。

In [6]:
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None

class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""

class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u

class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

In [7]:
prompt_tokens=tok.apply_chat_template([{'role':'user','content':'hello'}],
    tokenize=True,add_generation_prompt=True)
pool=PagedKVPool(8,4,models['stable'].cfg,set(models))
runner=CachedModelRunner(tok,models,pool)
for version in ['stable','canary']:
    sid='demonstrate_'+version
    runner.open_prefill(sid,version,prompt_tokens)
    print(version,'prefill与全前向最大误差:',runner.assert_consistent(sid))
    token=int(runner.last_logits[sid][16:].argmax())+16
    runner.step(sid,token)
    print(version,'decode后最大误差:',runner.assert_consistent(sid))
    runner.close(sid)
print('最终活跃物理块:',pool.audit()['live_blocks'])

stable prefill与全前向最大误差: 2.384185791015625e-07
stable decode后最大误差: 2.384185791015625e-07
canary prefill与全前向最大误差: 2.384185791015625e-07
canary decode后最大误差: 2.384185791015625e-07
最终活跃物理块: 0


## 3. 事故处理需要“请求级版本固定”和幂等键

一个准入请求携带可信租户、幂等键、文本、deadline 与停止预算。路由用租户与幂等键的确定性哈希，新的请求才按当前权重选版本。

同一租户同一幂等键的**完全相同请求**返回第一次的终态，不重复执行模型，也不重复计入终态分母。相同键不同参数明确报错。

这解释了回滚后的一个细节：同一个已完成请求的幂等重试仍可返回灰度旧结果；它不属于**回滚后新准入的稳定版本流量**，不能拿来伪装恢复样本。

本课没有真实HTTP服务、用户鉴权或分布式注册表。`tenant` 在此是由可信网关注入的教学字段；上线时不能直接相信来自客户端的租户声明。

In [8]:
@dataclass(frozen=True)
class ServiceRequest:
    tenant: str
    key: str
    text: str = 'hello'
    timeout_ticks: int = 70
    max_new_tokens: int = 3
    template: str = 'chat-v1'

    def validate(self):
        for name in ('tenant','key','template'):
            v=getattr(self,name)
            require(type(v) is str and bool(re.fullmatch(r'[A-Za-z0-9_-]{1,48}',v)),name+'格式非法')
        bounded_str(self.text,128,'text')
        require(bool(self.text),'请求正文不能为空')
        require(type(self.timeout_ticks) is int and 1<=self.timeout_ticks<=1000,'timeout_ticks非法')
        require(type(self.max_new_tokens) is int and 1<=self.max_new_tokens<=6,'max_new_tokens非法')

    def fingerprint(self):
        self.validate()
        return digest(asdict(self))


@dataclass
class Admission:
    version: str
    epoch: int
    fingerprint: str
    trace_id: str
    request: ServiceRequest
    terminal: dict | None = None
    completed: bool = False


class CanaryRouter:
    def __init__(self,stable='stable',canary='canary',weight=0):
        require(stable!=canary and all(type(v) is str and v for v in (stable,canary)),'版本非法')
        self.stable=stable;self.canary=canary
        self.weight=0;self.epoch=0;self.latched=False
        self.set_weight(weight)

    @staticmethod
    def bucket(tenant,key):
        return int(hashlib.sha256((tenant+'|'+key).encode('utf-8')).hexdigest()[:8],16)%100

    def choose(self,req:ServiceRequest):
        req.validate()
        return self.canary if self.bucket(req.tenant,req.key)<self.weight else self.stable

    def set_weight(self,weight):
        require(type(weight) is int and 0<=weight<=100,'灰度权重必须为0..100整数')
        require(not self.latched or weight==0,'回滚锁存后不得自动开启灰度')
        self.weight=weight

    def start_canary(self,weight):
        require(self.weight==0 and not self.latched,'已有灰度或处于回滚锁存状态')
        self.set_weight(weight)
        require(self.weight>0,'启用灰度权重必须为正')
        self.epoch+=1
        return self.epoch

    def rollback(self):
        if self.latched:return False
        require(self.weight>0,'没有灰度版本可以回滚')
        self.weight=0;self.latched=True;self.epoch+=1
        return True

    def manual_reenable(self,approved:bool,postcheck:bool,weight:int):
        require(approved is True and postcheck is True,'重新开启灰度需要人工批准和恢复验证')
        require(self.latched,'当前没有回滚锁存状态')
        require(type(weight) is int and 1<=weight<=100,'新灰度权重非法')
        self.latched=False;self.weight=weight;self.epoch+=1


class SafeAudit:
    """不记录正文或主观归因；哈希链检测偶发篡改，不是签名或不可抵赖机制。"""
    def __init__(self):
        self.rows=[]

    def append(self,event:str,details:dict):
        require(event in {'canary_start','alert','rollback','operator_page','postcheck','manual_reenable','release_leaks'},
                '审计事件类型非法')
        require(type(details) is dict and len(canonical(details))<=3000,'审计细节不合法')
        previous=self.rows[-1]['hash'] if self.rows else '0'*64
        row={'seq':len(self.rows),'event':event,'details':copy.deepcopy(details),'prev':previous}
        row['hash']=digest(row)
        self.rows.append(row)
        return row['hash']

    def verify(self):
        for i,row in enumerate(self.rows):
            require(row['seq']==i,'审计序号错位')
            require(row['prev']==(self.rows[i-1]['hash'] if i else '0'*64),'审计哈希链断裂')
            require(row['hash']==digest({k:v for k,v in row.items() if k!='hash'}),'审计记录被改写')
        return {'count':len(self.rows),'root_hash':self.rows[-1]['hash'] if self.rows else '0'*64}

In [9]:
router=CanaryRouter()
router.start_canary(50)
examples=[ServiceRequest('orgA',f'case{i:03d}') for i in range(12)]
print('路由样例（确定性哈希，不是随机投币）：')
print([(r.key,router.bucket(r.tenant,r.key),router.choose(r)) for r in examples])
old_epoch=router.epoch
print('首次回滚提交:',router.rollback(),'新的epoch:',router.epoch)
print('重复回滚提交:',router.rollback())
print('新请求在回滚后都进稳定版本:',all(router.choose(r)=='stable' for r in examples))

路由样例（确定性哈希，不是随机投币）：
[('case000', 2, 'canary'), ('case001', 9, 'canary'), ('case002', 44, 'canary'), ('case003', 59, 'stable'), ('case004', 80, 'stable'), ('case005', 17, 'canary'), ('case006', 43, 'canary'), ('case007', 52, 'stable'), ('case008', 12, 'canary'), ('case009', 90, 'stable'), ('case010', 28, 'canary'), ('case011', 32, 'canary')]
首次回滚提交: True 新的epoch: 2
重复回滚提交: False
新请求在回滚后都进稳定版本: True


## 4. 真正执行模型请求，同时插入**可识别**的故障

四种受控模式：

| 模式 | 行为 | 期望观测 |
|---|---|---|
| `none` | 正常真实 KV 推理 | 成功、KV释放 |
| `slow` | 仅灰度的逻辑耗时增加40 tick | 请求可完成但违反延迟 SLO |
| `global_slow` | stable、canary 都慢 | 可能是共享故障，不能无凭据自动归咎canary |
| `kv_leak` | 故意保留灰度完成会话的物理KV页 | 物理占用上升，回滚时需实际释放 |

还支持 `internal`：真实 prefill 后注入失败，故障要按内部错误终态落账并释放KV。

**错误不是被当成 EOS 或空回答。** 逻辑耗时、HTTP/真实wall-clock耗时、生成正确率是三个不同的量，本课只测前者和资源不变量。

In [10]:
@dataclass(frozen=True)
class DrillConfig:
    capacity_blocks: int=32
    block_size: int=4
    canary_weight: int=50
    slo_target: float=.95
    latency_target_ticks: int=25
    short_n: int=6
    long_n: int=12
    min_excess_bad: float=.25
    trigger_burn: float=4.0
    max_stable_bad: float=.20
    leak_pressure_fraction: float=.35
    leak_min_sessions: int=3
    max_version_staleness: int=12
    recover_samples: int=12

    def validate(self):
        ints=('capacity_blocks','block_size','latency_target_ticks','short_n','long_n',
              'leak_min_sessions','max_version_staleness','recover_samples')
        require(all(type(getattr(self,k)) is int and getattr(self,k)>0 for k in ints),'整数参数必须为正')
        require(type(self.canary_weight) is int and 1<=self.canary_weight<=99,'灰度权重限制1..99')
        require(self.long_n>=self.short_n>=2,'窗口长度顺序错误')
        for k in ('slo_target','min_excess_bad','max_stable_bad','leak_pressure_fraction'):
            v=getattr(self,k)
            require(type(v) in (float,int) and math.isfinite(v) and 0<v<1,k+'必须在0..1')
        require(type(self.trigger_burn) in (float,int) and math.isfinite(self.trigger_burn)
                and self.trigger_burn>0,'burn阈值错误')


class IncrementalGateway:
    """只有准入/逻辑时间模拟，KV与attention是真实CPU tensor。无HTTP、无网络和异步并发。"""
    def __init__(self,tok,models,config:DrillConfig):
        config.validate()
        require(set(models)=={'stable','canary'},'需要stable/canary两个版本')
        self.tok=tok;self.models=models;self.config=config
        self.router=CanaryRouter(weight=0)
        self.pool=PagedKVPool(config.capacity_blocks,config.block_size,
                              models['stable'].cfg,set(models))
        self.runner=CachedModelRunner(tok,models,self.pool)
        self.registry={}
        self.terminal=[]
        self.traces=[]
        self.logical_tick=0
        self.execution_count=0
        self.leaked_sessions={}
        self.baseline_hash=state_hash(models['stable'].state_dict())
        self.canary_hash=state_hash(models['canary'].state_dict())
        self.peak_live_blocks=0
        self.audit=SafeAudit()

    def admit(self,req:ServiceRequest):
        req.validate();identity=(req.tenant,req.key)
        fp=req.fingerprint()
        if identity in self.registry:
            old=self.registry[identity]
            require(old.fingerprint==fp,'同租户幂等键冲突：不得重新执行')
            return old,False
        version=self.router.choose(req)
        trace=digest(['trace',req.tenant,req.key])[:32]
        rec=Admission(version,self.router.epoch,fp,trace,req)
        self.registry[identity]=rec
        return rec,True

    def _safe_terminal(self,rec,status,latency,generated,kv_peak,cache_diff,stage):
        require(not rec.completed,'同一请求不允许重复终态')
        good=(status=='ok' and latency<=self.config.latency_target_ticks)
        row={'ordinal':len(self.terminal),'trace_id':rec.trace_id,
             'version':rec.version,'admitted_epoch':rec.epoch,
             'terminal_status':status,'duration_ticks':latency,'slo_good':good,
             'generated_tokens':generated,'kv_peak_blocks':kv_peak,
             'cache_full_max_difference':cache_diff,'stage':stage}
        self.terminal.append(row)
        rec.terminal=row;rec.completed=True
        # Trace ID仅作诊断连接键，不用于指标标签。原始文本/tenant/key不落日志。
        self.traces.append({'trace_id':rec.trace_id,'span':'inference','version':rec.version,
                            'stage':stage,'status':status,'duration_ticks':latency,
                            'prompt_digest':digest(rec.request.text)[:16]})
        return row

    def execute(self,rec:Admission,fault='none'):
        require(self.registry.get((rec.request.tenant,rec.request.key)) is rec,
                '请求不是当前服务正式准入的对象')
        if rec.completed:return rec.terminal,False
        require(fault in {'none','slow','internal','kv_leak','global_slow'},'非法故障模式')
        require(state_hash(self.models['stable'].state_dict())==self.baseline_hash,'稳定版本被改写')
        require(state_hash(self.models['canary'].state_dict())==self.canary_hash,'灰度版本被改写')
        self.execution_count+=1
        sid='req'+str(self.execution_count)
        prompt_ids=self.tok.apply_chat_template([{'role':'user','content':rec.request.text}],
                                                 tokenize=True,add_generation_prompt=True)
        require(len(prompt_ids)+rec.request.max_new_tokens<=self.models[rec.version].cfg.max_length,
                '上下文越界不静默截断')
        slow=(fault=='global_slow' or (fault=='slow' and rec.version=='canary'))
        deliberately_leak=(fault=='kv_leak' and rec.version=='canary')
        internal=(fault=='internal' and rec.version=='canary')
        latency=math.ceil(len(prompt_ids)/3)
        generated=0;max_diff=0.0
        status='ok';stage='prefill';opened=False;leaked=False
        try:
            self.runner.open_prefill(sid,rec.version,prompt_ids)
            opened=True
            max_diff=max(max_diff,self.runner.assert_consistent(sid))
            if internal:raise RuntimeError('injected_internal_failure')
            stage='decode'
            for _ in range(rec.request.max_new_tokens):
                logits=self.runner.last_logits[sid]
                # 屏蔽无效的结构token，避免把控制ID当作正文去解码。
                scores=logits.clone()
                scores[:16]=-torch.inf
                next_id=int(scores.argmax())
                self.runner.step(sid,next_id)
                generated+=1
                latency+=2
                if next_id==self.tok.eos_token_id:break
            max_diff=max(max_diff,self.runner.assert_consistent(sid))
            if slow:latency+=40  # 故意注入逻辑开销；不sleep、不声称真实RTT
            if latency>rec.request.timeout_ticks:
                status='deadline_exceeded'
            if deliberately_leak:
                self.leaked_sessions[sid]='canary'
                leaked=True
        except NoKVSpace:
            status='capacity_rejected';stage='kv_allocation'
        except RuntimeError as exc:
            require(str(exc)=='injected_internal_failure','未知运行错误必须向上抛出')
            status='internal_error';stage='model_runtime'
        finally:
            if sid in self.pool.sessions and not leaked:
                s=self.pool.sessions[sid]
                if s.pending is not None:self.pool.abort(sid)
                self.runner.close(sid)
            self.pool.audit()
        self.peak_live_blocks=max(self.peak_live_blocks,self.pool.usage()['live_blocks'])
        # 模拟中每个请求推进一个到达顺序tick，latency是单请求的独立逻辑工作量。
        self.logical_tick+=1
        return self._safe_terminal(rec,status,latency,generated,
                                   self.peak_live_blocks,max_diff,stage),True

    def serve(self,req:ServiceRequest,fault='none'):
        rec,new=self.admit(req)
        if not new and rec.completed:return rec.terminal,False
        return self.execute(rec,fault)

    def release_leaks(self):
        items=list(self.leaked_sessions)
        for sid in items:
            require(sid in self.pool.sessions,'泄漏记录找不到真实会话')
            self.runner.close(sid)
            del self.leaked_sessions[sid]
        self.pool.audit()
        return len(items)

    def metrics(self):
        counter=Counter((r['version'],r['terminal_status']) for r in self.terminal)
        # 低基数维度仅保留服务版本及枚举状态，不包含tenant、key、trace_id或文本。
        return {'terminal_counts':[{'version':v,'status':s,'count':n}
                                   for (v,s),n in sorted(counter.items())],
                'attempts':len(self.registry),'real_model_executions':self.execution_count,
                'kv_live_blocks':self.pool.usage()['live_blocks'],
                'kv_leaked_sessions':len(self.leaked_sessions),
                'kv_peak_live_blocks':self.peak_live_blocks,
                'kv_bytes_per_block':self.pool.bytes_per_block}

In [11]:
config=DrillConfig()
probe=IncrementalGateway(tok,models,config)
item=ServiceRequest('orgA','duplicate_0',text='hello')
first,did_execute=probe.serve(item)
again,second_execute=probe.serve(item)
print('首次真实推理:',did_execute,'再试是否重算:',second_execute)
print('结果对象相同:',first is again,'唯一终态:',len(probe.terminal))
print('本次增量KV与full误差:',first['cache_full_max_difference'])
print('资源最终状态:',probe.pool.usage()['live_blocks'])
print('低基数指标:',probe.metrics()['terminal_counts'])

首次真实推理: True 再试是否重算: False
结果对象相同: True 唯一终态: 1
本次增量KV与full误差: 2.384185791015625e-07
资源最终状态: 0
低基数指标: [{'version': 'stable', 'status': 'ok', 'count': 1}]


## 5. SLO 失败不等于 HTTP 5xx

本课采用示例 SLO：95% 的唯一终态请求需要成功，且模拟逻辑工作量不超过25 tick。

- 模型输出完成但用了50 tick：`terminal_status=ok`，**`slo_good=False`**。
- 内部错误、容量拒绝、超出deadline：也计入不良终态。
- 幂等重复请求：不增加终态分母。

错误预算率是 `1−0.95=0.05`。在一个窗口中，失败率为20%时，燃烧率是 `0.20/0.05=4` 倍。

Google SRE 的多窗口 burn-rate 告警使用**时间窗口**。本课为了在短离线实验中演示采样门槛，刻意采用最近6条、12条**按请求数排列的窗口**，它们不是 Google 原则的生产时间窗口或推荐默认阈值。[SRE Workbook](https://sre.google/workbook/alerting-on-slos/)

In [12]:
def window_sli(rows:list[dict],n:int,target:float):
    require(type(n) is int and n>0 and 0<target<1,'窗口/目标非法')
    chosen=rows[-n:]
    if len(chosen)<n:return {'available':False,'count':len(chosen),'required':n}
    bad=sum(not r['slo_good'] for r in chosen)
    return {'available':True,'count':n,'bad':bad,'good':n-bad,
            'bad_rate':bad/n,'burn':(bad/n)/(1-target)}


def assess_canary(gateway:IncrementalGateway, config:DrillConfig):
    """两个按请求数的窗口（非Google手册中的真实时间窗口）。
    使用当前灰度epoch的终态；必须有足够的稳定对照与灰度样本。
    """
    require(config==gateway.config,'评估配置必须与网关一致')
    epoch=gateway.router.epoch
    evidence={}
    for version in ('stable','canary'):
        data=[r for r in gateway.terminal if r['version']==version and r['admitted_epoch']==epoch]
        latest_age=(len(gateway.terminal)-1-data[-1]['ordinal']) if data else None
        evidence[version]={'short':window_sli(data,config.short_n,config.slo_target),
                           'long':window_sli(data,config.long_n,config.slo_target),
                           'latest_age_requests':latest_age}
    detail={'epoch':epoch,'weight':gateway.router.weight,'windows':evidence,
            'live_blocks':gateway.pool.usage()['live_blocks'],
            'leaked_canary_sessions':len(gateway.leaked_sessions)}
    if gateway.router.latched or gateway.router.weight==0:
        return {'action':'hold','reason':'no_active_canary','evidence':detail}
    leak_pressure=detail['live_blocks']/config.capacity_blocks
    if len(gateway.leaked_sessions)>=config.leak_min_sessions and leak_pressure>=config.leak_pressure_fraction:
        detail['leak_pressure_fraction']=leak_pressure
        return {'action':'rollback','reason':'confirmed_canary_kv_leak_pressure','evidence':detail}
    if not all(evidence[v][w]['available'] for v in ('stable','canary') for w in ('short','long')):
        return {'action':'wait','reason':'insufficient_per_version_samples','evidence':detail}
    if any(evidence[v]['latest_age_requests']>config.max_version_staleness for v in ('stable','canary')):
        return {'action':'wait','reason':'stale_version_telemetry','evidence':detail}
    ss=evidence['stable']['short']; sl=evidence['stable']['long']
    cs=evidence['canary']['short']; cl=evidence['canary']['long']
    if ss['burn']>=config.trigger_burn and sl['burn']>=config.trigger_burn:
        return {'action':'page','reason':'both_versions_or_stable_degraded','evidence':detail}
    threshold=(cs['burn']>=config.trigger_burn and cl['burn']>=config.trigger_burn
               and cs['bad_rate']-ss['bad_rate']>=config.min_excess_bad
               and ss['bad_rate']<=config.max_stable_bad)
    if threshold:
        return {'action':'rollback','reason':'canary_dual_window_burn','evidence':detail}
    return {'action':'hold','reason':'canary_gate_not_met','evidence':detail}


class IncidentController:
    def __init__(self,gateway:IncrementalGateway,known_good_hash:str):
        self.gateway=gateway;self.stable_hash=known_good_hash
        self.state='normal';self.transitions=[];self.last_decision=None
        self.recovered=False;self.terminal_rollback_hash=None

    def start(self,weight:int):
        require(self.state=='normal','灰度只能从normal启动')
        require(state_hash(self.gateway.models['stable'].state_dict())==self.stable_hash,
                '发布前稳定权重与已知良好身份不符')
        epoch=self.gateway.router.start_canary(weight)
        self.state='monitoring'
        self._record('canary_start',{'epoch':epoch,'weight':weight,'stable_hash':self.stable_hash})

    def _record(self,event,details):
        self.gateway.audit.append(event,details)
        self.transitions.append({'event':event,**copy.deepcopy(details)})

    def consider(self,decision:dict):
        require(type(decision) is dict and set(decision)=={'action','reason','evidence'},'决策字段错误')
        # 在应用前重新评估，防止延迟、篡改或旧epoch证据触发事故动作。
        expected=assess_canary(self.gateway,self.gateway.config)
        require(decision==expected,'报告不是当前最新窗口证据，不可执行')
        self.last_decision=copy.deepcopy(decision)
        if self.state in {'mitigating','stabilizing','resolved','needs_operator'}:
            return 'already_handled'
        require(self.state=='monitoring','只有monitoring允许评估')
        if decision['action']=='page':
            self.state='needs_operator'
            self._record('operator_page',{'epoch':self.gateway.router.epoch,
                                          'reason':decision['reason']})
            return 'needs_operator'
        if decision['action']!='rollback':return decision['action']
        self._record('alert',{'epoch':self.gateway.router.epoch,'reason':decision['reason'],
                              'canary_short':decision['evidence']['windows']['canary']['short'],
                              'canary_long':decision['evidence']['windows']['canary']['long']})
        require(state_hash(self.gateway.models['stable'].state_dict())==self.stable_hash,
                '无法验证已知良好版本，拒绝静默回滚')
        changed=self.gateway.router.rollback()
        require(changed,'同一回滚只允许提交一次')
        self.state='mitigating'
        self.terminal_rollback_hash=self.gateway.audit.append('rollback',
                      {'epoch':self.gateway.router.epoch,'stable_hash':self.stable_hash,
                       'canary_weight':self.gateway.router.weight})
        self.transitions.append({'event':'rollback','epoch':self.gateway.router.epoch,
                                 'canary_weight':self.gateway.router.weight})
        released=self.gateway.release_leaks()
        if released:self._record('release_leaks',{'released_sessions':released})
        require(self.gateway.pool.usage()['live_blocks']==0,'回滚后仍有遗留KV页，不能声称恢复')
        self.state='stabilizing'
        return 'rolled_back'

    def postcheck(self,new_rows:list[dict]):
        require(self.state=='stabilizing','没有进入恢复验证阶段')
        require(len(new_rows)>=self.gateway.config.recover_samples,'恢复验证样本不足')
        require(all(r['version']=='stable' and r['admitted_epoch']==self.gateway.router.epoch
                    for r in new_rows),'恢复验证必须只用回滚之后的新稳定版本流量')
        require(len({r['trace_id'] for r in new_rows})==len(new_rows),'不能把同一次请求重试当新样本')
        require(all(any(r is actual for actual in self.gateway.terminal) for r in new_rows),
                '恢复验证不能使用人工拼造的结果')
        require(not any(r.version=='canary' and not r.completed for r in self.gateway.registry.values()),
                '仍有旧灰度请求未结束，必须先排空或显式取消')
        ok=all(r['slo_good'] for r in new_rows) and not self.gateway.leaked_sessions
        ok=ok and self.gateway.pool.usage()['live_blocks']==0
        ok=ok and state_hash(self.gateway.models['stable'].state_dict())==self.stable_hash
        self._record('postcheck',{'samples':len(new_rows),'success':ok,'kv_live_blocks':
                                 self.gateway.pool.usage()['live_blocks']})
        self.state='resolved' if ok else 'needs_operator'
        self.recovered=ok
        return ok

    def manual_reenable(self,approved=False,weight=5):
        require(self.state=='resolved' and self.recovered,'需要先完成恢复验证')
        self.gateway.router.manual_reenable(approved,self.recovered,weight)
        self._record('manual_reenable',{'epoch':self.gateway.router.epoch,'weight':weight})
        self.state='monitoring';self.recovered=False

In [13]:
def find_keys(router:CanaryRouter,version:str,n:int,used:set[str],start=0,tenant='orgA'):
    require(version in {'stable','canary'} and type(n) is int and n>0,'目标版本/数量错误')
    keys=[];i=start
    while len(keys)<n:
        key=f'k{i:05d}';i+=1
        if key in used:continue
        if router.choose(ServiceRequest(tenant,key))==version:
            used.add(key);keys.append(key)
    return keys


def stage_traffic(gateway:IncrementalGateway,counts:dict,used:set,mode='none',tenant='orgA'):
    """控制组流量数量：用于可解释的配对故障演练，不代表真实在线流量分布。"""
    require(set(counts)<={'stable','canary'},'未知流量组')
    keys=[]
    for version,count in counts.items():
        if count:keys.extend((version,k) for k in find_keys(gateway.router,version,count,used,tenant=tenant))
    # 先交错排列，防止稳定版本全部排在最后后才进行观察。
    stable=[k for v,k in keys if v=='stable'];canary=[k for v,k in keys if v=='canary']
    ordered=[]
    for i in range(max(len(stable),len(canary))):
        if i<len(stable):ordered.append(('stable',stable[i]))
        if i<len(canary):ordered.append(('canary',canary[i]))
    rows=[]
    for expected_version,key in ordered:
        row,_=gateway.serve(ServiceRequest(tenant,key,text='hello',timeout_ticks=70),fault=mode)
        require(row['version']==expected_version,'灰度路由结果与键分桶不一致')
        rows.append(row)
    return rows

In [14]:
rows=[{'slo_good':True}]*8+[{'slo_good':False}]*2
print('10条中2条坏:',window_sli(rows,10,.95))
print('长窗口不足时必须等待:',window_sli(rows,12,.95))
print('0个坏请求:',window_sli([{'slo_good':True}]*12,12,.95))

10条中2条坏: {'available': True, 'count': 10, 'bad': 2, 'good': 8, 'bad_rate': 0.2, 'burn': 3.999999999999997}
长窗口不足时必须等待: {'available': False, 'count': 10, 'required': 12}
0个坏请求: {'available': True, 'count': 12, 'bad': 0, 'good': 12, 'bad_rate': 0.0, 'burn': 0.0}


## 6. 如何避免“一个慢请求就自动回滚”？

本课要求每个版本在近期窗口里都达到最小样本量：stable short≥6、long≥12，canary同样如此；任一版本最后一条终态距离当前超过12条请求时，视为遥远的历史证据，不用于回滚。只有双窗口都超过示例 burn 阈值，且 canary 比 stable 明显更差、稳定版本仍健康，才自动关闭新的灰度路由。

如果 stable 和 canary **同时**严重变慢，控制器标记 `needs_operator`，产生人工处理审计，而不是假设一键回滚一定有效。

这是**保守的教学决策规则**，不是自动根因分析器；它不能证明故障因果。负载也使用平衡构造的 stable/canary 两组请求，便于对照，不能将其当真实生产流量随机化实验。

In [15]:
gate=IncrementalGateway(tok,models,config)
control=IncidentController(gate,gate.baseline_hash)
control.start(50)
used=set()
stage_traffic(gate,{'stable':6,'canary':6},used,'slow')
first_decision=assess_canary(gate,config)
print('第一阶段:',first_decision['action'],first_decision['reason'])
stage_traffic(gate,{'stable':6,'canary':6},used,'slow')
second_decision=assess_canary(gate,config)
print('第二阶段:',second_decision['action'],second_decision['reason'])
print('灰度短窗口:',second_decision['evidence']['windows']['canary']['short'])
print('稳定短窗口:',second_decision['evidence']['windows']['stable']['short'])

第一阶段: wait insufficient_per_version_samples


第二阶段: rollback canary_dual_window_burn
灰度短窗口: {'available': True, 'count': 6, 'bad': 6, 'good': 0, 'bad_rate': 1.0, 'burn': 19.999999999999982}
稳定短窗口: {'available': True, 'count': 6, 'bad': 0, 'good': 6, 'bad_rate': 0.0, 'burn': 0.0}


## 7. 事故控制器不是一段 `if bad: rollback()`

执行回滚前还需要：

1. 用最新观测重新生成决策，拒绝过期/篡改的窗口证据。
2. 验证稳定版本的已知良好权重身份没有改变。
3. 将**新准入** canary 权重降至0，并锁存，禁止自动反弹。
4. 已准入 canary 保持原版本或显式排空；不能中途换 adapter。
5. 处理此次已知的灰度KV会话泄漏，真实释放物理页。
6. 使用**回滚后新的稳定请求**做足量恢复验证；否则不得宣布解决。

审计日志记录事件序号、前一条哈希与当前内容哈希。它只帮助发现偶发修改，**不是数字签名，更不能抵御能改写整条链的攻击者**。

真实事故还需要指挥、操作、沟通和事后分析的职责分离；这里仅模拟技术控制器。[Google SRE：Managing Incidents](https://sre.google/sre-book/managing-incidents/)

In [16]:
print('控制器在相同最新证据上执行:',control.consider(second_decision))
print('路由灰度权重:',gate.router.weight,'状态:',control.state)
post=stage_traffic(gate,{'stable':config.recover_samples},used,'none')
print('回滚后稳定流量验证:',control.postcheck(post),'状态:',control.state)
print('审计记录数与root:',gate.audit.verify())

控制器在相同最新证据上执行: rolled_back
路由灰度权重: 0 状态: stabilizing


回滚后稳定流量验证: True 状态: resolved
审计记录数与root: {'count': 4, 'root_hash': '83ca1be938183d23a9330efbeb782c3253c41889a50a028d322d71fd15456110'}


## 8. 四个独立事故场景，验证不同的**正确动作**

- **健康灰度**：不触发回滚，持续观察。
- **灰度慢请求**：stable 对照健康，灰度双窗口违规→回滚→重新验证。
- **全局共享变慢**：两版本都差→上报人工处置；不能冒称回滚成功。
- **灰度 KV 泄漏**：物理占用上升，即使尚未造成延迟失败，也可以基于明确的泄漏压力信号隔离灰度、回收页并验证。

尤其注意：KV 泄漏触发的不是同一个 SLO 规则；物理压力门槛是这次额外的**资源安全信号**。报告会保留触发原因，而不是把所有动作都标成“因 SLO 回滚”。

In [17]:
def incident_scenario(name:str,tok,models,cfg:DrillConfig):
    require(name in {'healthy','canary_slow','global_slow','canary_kv_leak'},'未知事故场景')
    gateway=IncrementalGateway(tok,models,cfg)
    controller=IncidentController(gateway,state_hash(models['stable'].state_dict()))
    used=set()
    warm=stage_traffic(gateway,{'stable':8},used)
    require(all(r['slo_good'] for r in warm),'健康基线必须先建立')
    controller.start(cfg.canary_weight)
    mode={'healthy':'none','canary_slow':'slow','global_slow':'global_slow',
          'canary_kv_leak':'kv_leak'}[name]
    samples=[];decisions=[];last='wait'
    for phase in range(4):
        block=stage_traffic(gateway,{'stable':6,'canary':6},used,mode)
        samples.extend(block)
        decision=assess_canary(gateway,cfg)
        decisions.append({'phase':phase+1,'result':copy.deepcopy(decision)})
        last=controller.consider(decision)
        if last in {'rolled_back','needs_operator'}:break
    recovery=[]
    if last=='rolled_back':
        recovery=stage_traffic(gateway,{'stable':cfg.recover_samples},used,'none')
        controller.postcheck(recovery)
    if gateway.leaked_sessions:
        # 仅演示结束的最终清理；不掩盖未恢复的状态。
        gateway.release_leaks()
    require(gateway.pool.usage()['live_blocks']==0,'场景结束时KV内存泄漏')
    gateway.pool.audit();gateway.audit.verify()
    # 提取可用于判断归因的关键内容，报告不包含用户正文或原始tenant/key。
    report={'scenario':name,'initial_baseline_successes':sum(r['slo_good'] for r in warm),
            'predecision_requests':len(samples),'stage_sample_versions':Counter(r['version'] for r in samples),
            'stage_slo_bad_versions':Counter(r['version'] for r in samples if not r['slo_good']),
            'decisions':decisions,'last_controller_result':last,
            'controller_state':controller.state,'recovered':controller.recovered,
            'router_weight_after':gateway.router.weight,'router_epoch':gateway.router.epoch,
            'recovery_samples':len(recovery),'recovery_bad':sum(not r['slo_good'] for r in recovery),
            'metrics':gateway.metrics(),'audit_chain':gateway.audit.rows,
            'audit_summary':gateway.audit.verify(),
            'telemetry':gateway.terminal,'traces':gateway.traces,
            'limitations':['synthetic arrival order and logical duration ticks',
                           'balanced synthetic canary/stable cohorts, not observed production traffic',
                           'random initialized model; no semantic correctness evaluation',
                           'not OpenTelemetry/Prometheus/real autoscaling; no live traffic or network']}
    return report


def save_models(folder,tok,models):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    stable=models['stable']
    cfg=asdict(stable.cfg)
    foundation={k:t.cpu().detach().clone() for k,t in base_state(stable).items()}
    torch.save({'version':VERSION,'kind':'base','config':cfg,'tokenizer':tok.contract(),
                'state':foundation,'base_hash':state_hash(foundation)},folder/'base.pt')
    for key,model in models.items():
        torch.save({'version':VERSION,'kind':'adapter','name':key,'config':cfg,
                    'base_hash':state_hash(foundation),'tokenizer':tok.contract(),
                    'rank':4,'alpha':8.0,'state':adapter_snapshot(model),
                    'model_hash':state_hash(model.state_dict())},folder/(key+'.pt'))


def load_models(folder):
    folder=Path(folder)
    base=torch.load(folder/'base.pt',map_location='cpu',weights_only=True)
    require(base['version']==VERSION and base['kind']=='base','基座版本/类型错误')
    tok=ByteMergeChatTokenizer()
    require(base['tokenizer']==tok.contract(),'词表身份不匹配')
    require(state_hash(base['state'])==base['base_hash'],'基座内容摘要不匹配')
    reference=TinyChatLM(ToyConfig(**base['config']))
    reference.load_state_dict(base['state'],strict=True)
    reference.eval().requires_grad_(False)
    models={}
    for name in ('stable','canary'):
        value=torch.load(folder/(name+'.pt'),map_location='cpu',weights_only=True)
        require(value['version']==VERSION and value['kind']=='adapter' and value['name']==name,'adapter类型/版本错误')
        require(value['config']==base['config'] and value['base_hash']==base['base_hash']
                and value['tokenizer']==tok.contract(),'adapter不是对应基座/词表')
        policy=inject_lora(reference,value['rank'],value['alpha'])
        restore_adapter(policy,value['state'])
        policy.eval().requires_grad_(False)
        require(state_hash(policy.state_dict())==value['model_hash'],'adapter权重内容摘要错误')
        models[name]=policy
    return tok,models


def reload_run(folder):
    folder=Path(folder);tok,models=load_models(folder)
    saved=json.loads((folder/'scenario_reports.json').read_text(encoding='utf-8'))
    cfg=DrillConfig(**json.loads((folder/'settings.json').read_text(encoding='utf-8'))['drill_config'])
    # 独立模型对象，执行同一确定性事故演练；不是精确续训。
    rerun={name:incident_scenario(name,tok,models,cfg) for name in saved}
    require(rerun==saved,'独立重载的事故/恢复结果不一致')
    return {'reports_equal':True,'scenario_count':len(saved),'models':{
        name:state_hash(model.state_dict()) for name,model in models.items()},
        'all_sessions_released':all(r['metrics']['kv_live_blocks']==0 for r in rerun.values()),
        'scope':'independent deterministic CPU replay; not real network or training checkpoint'}


def run_experiment(output_root=Path('lesson_79_outputs'),seed=7901):
    require(type(seed) is int and seed>=0,'seed非法')
    torch.set_num_threads(1)
    path=Path(output_root)/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%f')
    path.mkdir(parents=True,exist_ok=False)
    tok,models=build_servable_models(seed)
    cfg=DrillConfig();cfg.validate()
    write_json(path/'settings.json',{'version':VERSION,'seed':seed,'drill_config':asdict(cfg),
                                      'torch':str(torch.__version__),'python':platform.python_version(),
                                      'sources':SOURCES})
    save_models(path,tok,models)
    names=['healthy','canary_slow','global_slow','canary_kv_leak']
    scenarios={name:incident_scenario(name,tok,models,cfg) for name in names}
    write_json(path/'scenario_reports.json',scenarios)
    summary={name:{'status':r['controller_state'],'last_action':r['last_controller_result'],
                   'recovered':r['recovered'],'bad':dict(r['stage_slo_bad_versions']),
                   'kv_peak_blocks':r['metrics']['kv_peak_live_blocks'],
                   'kv_final_blocks':r['metrics']['kv_live_blocks']} for name,r in scenarios.items()}
    write_json(path/'summary.json',summary)
    write_json(path/'reload.json',reload_run(path))
    for name,row in summary.items():print(name,canonical(row))
    print('输出目录:',path.resolve())
    return path

In [18]:
individual_scenarios={}
for scenario_name in ['healthy','canary_slow','global_slow','canary_kv_leak']:
    report=incident_scenario(scenario_name,tok,models,config)
    individual_scenarios[scenario_name]=report
    print(scenario_name,{
        'decision':report['last_controller_result'],
        'state':report['controller_state'],
        'recovered':report['recovered'],
        'bad_by_version':dict(report['stage_slo_bad_versions']),
        'peak_kv_blocks':report['metrics']['kv_peak_live_blocks'],
        'final_kv_blocks':report['metrics']['kv_live_blocks']})

healthy {'decision': 'hold', 'state': 'monitoring', 'recovered': False, 'bad_by_version': {}, 'peak_kv_blocks': 0, 'final_kv_blocks': 0}


canary_slow {'decision': 'rolled_back', 'state': 'resolved', 'recovered': True, 'bad_by_version': {'canary': 12}, 'peak_kv_blocks': 0, 'final_kv_blocks': 0}
global_slow {'decision': 'needs_operator', 'state': 'needs_operator', 'recovered': False, 'bad_by_version': {'stable': 12, 'canary': 12}, 'peak_kv_blocks': 0, 'final_kv_blocks': 0}


canary_kv_leak {'decision': 'rolled_back', 'state': 'resolved', 'recovered': True, 'bad_by_version': {}, 'peak_kv_blocks': 18, 'final_kv_blocks': 0}


## 9. 读诊断信息，不要靠请求明文定位事故

本课 Trace 与低基数指标只包含有限枚举字段：版本、阶段、状态、逻辑耗时、trace ID 和提示摘要。没有把租户/幂等键/原始正文作为指标标签。

可从 `trace_id` 关联这一请求的终态和诊断事件。**文本摘要不是匿名化保证**：低熵文本可能被穷举，所以生产上还需脱敏、访问控制与保留期限。

下面列出故障相关的前两条 Trace 和审计事件类型。它们来自本地真实演练，不是网络抓包或生产 OpenTelemetry exporter。OpenTelemetry 的 Trace Context 传播是另一个独立领域。[官方概念](https://opentelemetry.io/docs/concepts/context-propagation/)

In [19]:
for name in ('canary_slow','global_slow','canary_kv_leak'):
    report=individual_scenarios[name]
    print(name,'审计事件:',[x['event'] for x in report['audit_chain']])
    print('前两条Trace:',report['traces'][:2])

canary_slow 审计事件: ['canary_start', 'alert', 'rollback', 'postcheck']
前两条Trace: [{'trace_id': 'cc8cdc022f20f550746a70e8c47f036c', 'span': 'inference', 'version': 'stable', 'stage': 'decode', 'status': 'ok', 'duration_ticks': 9, 'prompt_digest': '5aa762ae383fbb72'}, {'trace_id': '3d6e5284989f5ea28daa2ca7313209f3', 'span': 'inference', 'version': 'stable', 'stage': 'decode', 'status': 'ok', 'duration_ticks': 9, 'prompt_digest': '5aa762ae383fbb72'}]
global_slow 审计事件: ['canary_start', 'operator_page']
前两条Trace: [{'trace_id': 'cc8cdc022f20f550746a70e8c47f036c', 'span': 'inference', 'version': 'stable', 'stage': 'decode', 'status': 'ok', 'duration_ticks': 9, 'prompt_digest': '5aa762ae383fbb72'}, {'trace_id': '3d6e5284989f5ea28daa2ca7313209f3', 'span': 'inference', 'version': 'stable', 'stage': 'decode', 'status': 'ok', 'duration_ticks': 9, 'prompt_digest': '5aa762ae383fbb72'}]
canary_kv_leak 审计事件: ['canary_start', 'alert', 'rollback', 'release_leaks', 'postcheck']
前两条Trace: [{'trace_id': 'cc8

## 10. 回滚不是恢复：状态、证据与“不能假装成功”的情况

`rolled_back` 仅表示新灰度准入被关闭。还需要证明：

- 新到的请求**只路由稳定版本**，不是旧幂等结果。
- 至少12条新稳定流量完成，SLO全部良好。
- 缓存真正释放，物理块与会话数回零。
- 稳定版本权重身份仍匹配已知良好快照。
- 没有尚未完成的旧灰度请求。

否则状态转 `needs_operator`，不能用“回滚 API 返回成功”宣布事故消除。重新开灰度需要人工批准与完成验证，不能让健康信号自动翻转锁存。

CodeDeploy 的正式回滚也是一次单独的部署操作，可能由失败或阈值触发；这不是对恢复效果的自动保证。[AWS CodeDeploy 官方文档](https://docs.aws.amazon.com/codedeploy/latest/userguide/deployments-rollback-and-redeploy.html)

In [20]:
leak_report=individual_scenarios['canary_kv_leak']
print('峰值真实物理KV页:',leak_report['metrics']['kv_peak_live_blocks'])
print('最终物理KV页:',leak_report['metrics']['kv_live_blocks'])
print('恢复检查样本数:',leak_report['recovery_samples'])
print('回滚审计是否包含释放证据:',any(x['event']=='release_leaks' for x in leak_report['audit_chain']))
print('健康灰度无需触发回滚:',individual_scenarios['healthy']['router_weight_after'])

峰值真实物理KV页: 18
最终物理KV页: 0
恢复检查样本数: 12
回滚审计是否包含释放证据: True
健康灰度无需触发回滚: 50


## 11. SLO 的统计分母也会影响事故判断

如果一个客户端对同一幂等键重试五次，而服务只真正执行一次，**不要将这五次都当作五个独立成功请求**来稀释错误率。这里统计的是准入后完成的**唯一请求终态**，不是网络连接或重试尝试次数。

如果某请求服务状态正常但超过本课延迟 SLO，它仍是一个不良终态；不能因为 HTTP 200 就把它记为好。

生产场景需要明确定义：客户主动取消算不算 SLI 分母？超时是端到端还是仅服务段？谁是可信时间源？本课没有实现真实连接取消、分布式排队和wall-clock，因此**不以它回答生产服务的 SLO 达标率**。

In [21]:
slow_gateway=IncrementalGateway(tok,models,config)
slow_gateway.router.start_canary(50)
key=find_keys(slow_gateway.router,'canary',1,set())[0]
first,_=slow_gateway.serve(ServiceRequest('orgA',key),fault='slow')
print('终态状态:',first['terminal_status'],'SLO好坏:',first['slo_good'],'逻辑耗时:',first['duration_ticks'])
for _ in range(5):
    rerun,executed=slow_gateway.serve(ServiceRequest('orgA',key),fault='slow')
    assert rerun is first and not executed
print('实际模型执行次数:',slow_gateway.execution_count,'SLI终态分母:',len(slow_gateway.terminal))

终态状态: ok SLO好坏: False 逻辑耗时: 49
实际模型执行次数: 1 SLI终态分母: 1


## 12. 保存与独立重载：验证的是**事故回放的一致性**

完整运行输出：

| 文件 | 内容 |
|---|---|
| `base.pt` | 固定基座权重、结构与词表身份 |
| `stable.pt`、`canary.pt` | 两个不同的 LoRA adapter，与基座摘要绑定 |
| `scenario_reports.json` | 所有窗口、状态机、终态、Trace、KV与审计原始结果 |
| `summary.json` | 四种故障的回滚/人工处理结果摘要 |
| `settings.json` | 模型与实验设置、可追溯参考资料 |
| `reload.json` | 独立新对象加载后的可重复事故回放 |
| `checks.json` | 162项检查及其名称 |

注意：这里没有优化器和训练，也没有在真实服务执行发布。`weights_only=True` 只能减少某类 pickle 风险；仍只应加载可信文件。

In [22]:
RUN_FULL_DRILLS = True
RUN_DIR = None
if RUN_FULL_DRILLS:
    RUN_DIR = run_experiment(Path('lesson_79_outputs'), seed=7901)
else:
    print('本次完整四场景与磁盘重载已显式跳过；不生成模拟成功报告。')

healthy {"bad":{},"kv_final_blocks":0,"kv_peak_blocks":0,"last_action":"hold","recovered":false,"status":"monitoring"}
canary_slow {"bad":{"canary":12},"kv_final_blocks":0,"kv_peak_blocks":0,"last_action":"rolled_back","recovered":true,"status":"resolved"}
global_slow {"bad":{"canary":12,"stable":12},"kv_final_blocks":0,"kv_peak_blocks":0,"last_action":"needs_operator","recovered":false,"status":"needs_operator"}
canary_kv_leak {"bad":{},"kv_final_blocks":0,"kv_peak_blocks":18,"last_action":"rolled_back","recovered":true,"status":"resolved"}
输出目录: /mnt/data/_lesson79_build/notebook_work/lesson_79_outputs/run_20261105_053541_339183


In [23]:
if RUN_DIR is not None:
    saved_summary=json.loads((RUN_DIR/'summary.json').read_text(encoding='utf-8'))
    print('四个场景的正式保存摘要：')
    for name,info in saved_summary.items():print(name,info)
    print('重载校验:',json.loads((RUN_DIR/'reload.json').read_text(encoding='utf-8'))['reports_equal'])

四个场景的正式保存摘要：
canary_kv_leak {'bad': {}, 'kv_final_blocks': 0, 'kv_peak_blocks': 18, 'last_action': 'rolled_back', 'recovered': True, 'status': 'resolved'}
canary_slow {'bad': {'canary': 12}, 'kv_final_blocks': 0, 'kv_peak_blocks': 0, 'last_action': 'rolled_back', 'recovered': True, 'status': 'resolved'}
global_slow {'bad': {'canary': 12, 'stable': 12}, 'kv_final_blocks': 0, 'kv_peak_blocks': 0, 'last_action': 'needs_operator', 'recovered': False, 'status': 'needs_operator'}
healthy {'bad': {}, 'kv_final_blocks': 0, 'kv_peak_blocks': 0, 'last_action': 'hold', 'recovered': False, 'status': 'monitoring'}
重载校验: True


## 13. 自动检查：从“伪造准入”到“假恢复”

固定检查关注的不只是结果：

- 因果 logits /真实 KV 一致，跨版本KV共享被拒绝。
- 幂等键冲突不重复计算；慢成功不能充作SLO成功。
- 仅短窗口有证据时不会回滚；灰度流量长期缺席导致窗口过期时只等待；双版本全差不会盲目回滚。
- 陈旧epoch/被篡改窗口证据不能触发新的发布动作。
- 旧灰度请求已经准入时保持版本，排空前不得声称恢复。
- 泄漏分配的是**真实页**，释放后才能恢复；审计改写不能无声通过。
- adapter 重载要求对应的基座与不可变模型身份。

独立命令 `python ...py --checks-only` 执行同一组测试，不调用默认实验。

In [24]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,result):
        if not bool(result):raise AssertionError(name)
        self.names.append(name)
    def equal(self,name,left,right):
        self.ok(name,left==right)
    def close(self,name,left,right,tol=1e-8):
        self.ok(name,abs(float(left)-float(right))<=tol)
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,ValueError,TypeError):self.names.append(name);return
        raise AssertionError(name+': 错误操作被接受')


def run_checks():
    torch.set_num_threads(1)
    c=Checks();tok,models=build_servable_models()
    cfg=DrillConfig();cfg.validate()
    c.equal('tokenizer固定词表',tok.vocab_size,281)
    c.equal('UTF8正文往返',tok.decode(tok.encode('hello 42')), 'hello 42')
    ids=tok.apply_chat_template([{'role':'user','content':'hello'}],add_generation_prompt=True)
    c.ok('chat模板有BOS和assistant起始',ids[0]==tok.bos_token_id and ids[-1]==tok.role_ids['assistant'])
    c.ok('仅控制token占特殊ID',all(i>=16 for i in tok.encode('hello')))
    c.rejects('缺少正文拒绝',lambda:ServiceRequest('orgA','key1',text='').validate())
    c.rejects('不安全租户字符拒绝',lambda:ServiceRequest('evil/x','key1').validate())
    c.rejects('请求长度负数拒绝',lambda:ServiceRequest('orgA','key1',max_new_tokens=0).validate())
    c.rejects('超长文本拒绝',lambda:ServiceRequest('orgA','key1',text='X'*129).validate())
    c.ok('模型全部参数冻结',all(not p.requires_grad for m in models.values() for p in m.parameters()))
    c.ok('LoRA基座完全相同',base_hash(models['stable'])==base_hash(models['canary']))
    c.ok('LoRA版本不完全相同',state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()))
    c.ok('每个版本有多层真实K和V张量',models['stable'].cfg.layers>0 and models['stable'].cfg.heads>0)
    c.ok('两个模型对象不共享参数存储',all(a.data_ptr()!=b.data_ptr() for a,b in
          zip(models['stable'].parameters(),models['canary'].parameters())))
    pool=PagedKVPool(8,4,models['stable'].cfg,set(models))
    runner=CachedModelRunner(tok,models,pool)
    for version in ('stable','canary'):
        sid='check_'+version
        runner.open_prefill(sid,version,ids)
        c.ok('真实KV逐步推理与full一致_'+version,runner.assert_consistent(sid)<2e-5)
        nxt=int(runner.last_logits[sid][16:].argmax())+16
        runner.step(sid,nxt)
        c.ok('decode增量KV与full一致_'+version,runner.assert_consistent(sid)<2e-5)
        c.equal('cache版本固定_'+version,pool.sessions[sid].version,version)
        runner.close(sid)
    c.equal('全部推理完成后KV清零',pool.usage()['live_blocks'],0)
    c.equal('KV释放后会话为空',len(pool.sessions),0)
    c.ok('真实KV写入token计数',runner.tokens_processed>=2*len(ids))
    c.ok('物理页容量字节与公式一致',pool.bytes_per_block ==
          pool.block_size*2*pool.cfg.layers*pool.cfg.heads*pool.head_dim*pool.data.element_size())
    c.rejects('不存在会话不能进行KV步进',lambda:runner.step('notexist',18))
    pool.open('forkparent','stable')
    c.rejects('跨版本不允许共享KV',lambda:pool.fork('forkparent','child','canary'))
    pool.release('forkparent')
    c.equal('跨版本fork拒绝后仍无泄漏',pool.usage()['live_blocks'],0)

    router=CanaryRouter(weight=50)
    candidate=ServiceRequest('orgA','stable_key','hello')
    chosen=router.choose(candidate)
    c.ok('版本路由仅两值',chosen in {'stable','canary'})
    c.equal('路由是确定性',router.choose(candidate),chosen)
    c.ok('相同幂等键不同租户哈希不同',
         digest(['orgA','key'])!=digest(['orgB','key']))
    c.equal('100%前只可能按桶',router.bucket('orgA','foo'),router.bucket('orgA','foo'))
    c.rejects('灰度负权重拒绝',lambda:router.set_weight(-1))
    c.rejects('灰度权重浮点拒绝',lambda:router.set_weight(10.5))
    c.rejects('无灰度时回滚拒绝',lambda:CanaryRouter().rollback())
    c.equal('启动canary增加路由epoch',CanaryRouter().start_canary(50),1)
    c.rejects('已有灰度不得重复启动',lambda:router.start_canary(10))
    c.ok('路由回滚返回一次变更',router.rollback())
    c.ok('路由回滚锁存且权重归零',router.latched and router.weight==0)
    c.ok('重复路由回滚不重复改变',router.rollback() is False)
    c.rejects('锁存后拒绝直接加灰度权重',lambda:router.set_weight(10))
    c.rejects('锁存后拒绝未经批准的恢复',lambda:router.manual_reenable(False,True,5))
    c.rejects('锁存后拒绝未做验证的恢复',lambda:router.manual_reenable(True,False,5))
    router.manual_reenable(True,True,5)
    c.ok('人工批准后可重新开启低权重',router.weight==5 and not router.latched)

    gateway=IncrementalGateway(tok,models,cfg)
    req=ServiceRequest('orgA','retry0','hello')
    rec,new=gateway.admit(req)
    c.ok('首次准入是新请求',new and not rec.completed)
    first,executed=gateway.execute(rec)
    c.ok('第一次实际执行',executed and first['version']=='stable')
    pre=gateway.execution_count
    again,executed2=gateway.serve(req)
    c.ok('同一幂等键不重复执行',again is first and not executed2 and gateway.execution_count==pre)
    c.equal('重试不增加SLO分母',len(gateway.terminal),1)
    c.equal('重试不增加trace',len(gateway.traces),1)
    c.rejects('同键不同内容必须冲突',lambda:gateway.serve(ServiceRequest('orgA','retry0','changed')))
    c.rejects('伪造准入对象拒绝',lambda:gateway.execute(copy.deepcopy(rec)))
    c.ok('正常服务无KV泄漏',gateway.pool.usage()['live_blocks']==0)
    c.ok('一次真实请求有token与合法耗时',first['generated_tokens']>=1 and first['duration_ticks']>0)
    c.ok('真实cache与full最大差小',first['cache_full_max_difference']<2e-5)
    c.ok('Trace包含低基数版本状态',set(gateway.traces[0])==
         {'trace_id','span','version','stage','status','duration_ticks','prompt_digest'})
    c.ok('日志不含请求明文',all('hello' not in canonical(t) for t in gateway.traces))
    c.ok('指标标签不含请求key或用户输入',not any('retry0' in canonical(x) for x in gateway.metrics()['terminal_counts']))
    c.equal('指标总请求数为唯一键',gateway.metrics()['attempts'],1)
    c.equal('指标模型执行次数',gateway.metrics()['real_model_executions'],1)
    internal_gateway=IncrementalGateway(tok,models,cfg);internal_gateway.router.start_canary(50)
    keys=set(); key=find_keys(internal_gateway.router,'canary',1,keys)[0]
    broken,_=internal_gateway.serve(ServiceRequest('orgA',key),fault='internal')
    c.equal('故意内部错误明确记为失败','internal_error',broken['terminal_status'])
    c.ok('内部故障记录阶段',broken['stage']=='model_runtime')
    c.equal('内部故障KV全部释放',internal_gateway.pool.usage()['live_blocks'],0)
    normal_gateway=IncrementalGateway(tok,models,cfg);normal_gateway.router.start_canary(50)
    key=find_keys(normal_gateway.router,'canary',1,set())[0]
    slow,_=normal_gateway.serve(ServiceRequest('orgA',key),fault='slow')
    c.ok('慢请求可HTTP完成但SLO失败',slow['terminal_status']=='ok' and not slow['slo_good'])
    c.ok('慢请求逻辑时间超过阈值',slow['duration_ticks']>cfg.latency_target_ticks)
    timeout,_=normal_gateway.serve(ServiceRequest('orgA',find_keys(normal_gateway.router,'canary',1,{key})[0],
                                                  timeout_ticks=5),fault='slow')
    c.ok('硬deadline明确归类',timeout['terminal_status']=='deadline_exceeded' and not timeout['slo_good'])
    c.equal('慢请求后KV页释放',normal_gateway.pool.usage()['live_blocks'],0)
    c.rejects('未知故障模式拒绝',lambda:normal_gateway.serve(ServiceRequest('orgA','newz'),fault='panic'))
    c.equal('窗口不足时不可当作低错误率',window_sli([first],6,.95)['available'],False)
    exact=[{'slo_good':True} for _ in range(8)]+[{'slo_good':False} for _ in range(2)]
    ws=window_sli(exact,10,.95)
    c.equal('窗口分母按唯一终态记录',ws['count'],10)
    c.equal('窗口2个坏请求',ws['bad'],2)
    c.close('20%失败对应4倍burn',ws['burn'],4.0)
    c.close('尾部6请求恰有2坏时burn',window_sli(exact,6,.95)['burn'],2/6/.05)
    c.rejects('目标100%不可用于错误预算除法',lambda:window_sli(exact,6,1.0))
    c.rejects('空窗口不能作为足量',lambda:window_sli(exact,0,.95))
    c.rejects('burn配置太小拒绝',lambda:DrillConfig(trigger_burn=0).validate())
    c.rejects('short比long长拒绝',lambda:DrillConfig(short_n=13,long_n=12).validate())
    c.rejects('无效阈值拒绝',lambda:DrillConfig(slo_target=1).validate())

    healthy=incident_scenario('healthy',tok,models,cfg)
    bad=incident_scenario('canary_slow',tok,models,cfg)
    global_report=incident_scenario('global_slow',tok,models,cfg)
    leak=incident_scenario('canary_kv_leak',tok,models,cfg)
    c.equal('健康canary不被无故回滚',healthy['controller_state'],'monitoring')
    c.equal('健康canary保持原权重',healthy['router_weight_after'],cfg.canary_weight)
    c.ok('健康场景仍经过多个完整窗口',len(healthy['decisions'])>=2)
    c.ok('正常双窗口不触发',all(d['result']['action'] in ('wait','hold') for d in healthy['decisions']))
    c.equal('canary慢故障触发回滚',bad['last_controller_result'],'rolled_back')
    c.equal('回滚后需要恢复验证',bad['controller_state'],'resolved')
    c.ok('慢故障明确归因到灰度',bad['stage_slo_bad_versions'].get('canary',0)>=12 and
         bad['stage_slo_bad_versions'].get('stable',0)==0)
    c.equal('canary回滚后新流量权重为零',bad['router_weight_after'],0)
    c.ok('恢复验证有完整样本',bad['recovery_samples']>=cfg.recover_samples)
    c.equal('回滚后的稳定流量无SLO违规',bad['recovery_bad'],0)
    c.ok('回滚产生完整审计',{'alert','rollback','postcheck'}<=set(x['event'] for x in bad['audit_chain']))
    c.equal('双版本全局变慢要人工处理',global_report['controller_state'],'needs_operator')
    c.equal('全局变慢不得盲目回滚',global_report['router_weight_after'],cfg.canary_weight)
    c.ok('全局变慢有独立稳定版本失败证据',global_report['stage_slo_bad_versions'].get('stable',0)>0)
    c.ok('全局事故需要operator_page审计',any(x['event']=='operator_page' for x in global_report['audit_chain']))
    c.equal('KV泄漏路径触发回滚',leak['last_controller_result'],'rolled_back')
    c.ok('KV泄漏真实占用页增长',leak['metrics']['kv_peak_live_blocks']>0)
    c.ok('泄漏回滚具有物理回收证据',any(x['event']=='release_leaks' for x in leak['audit_chain']))
    c.equal('KV泄漏回滚后零占用',leak['metrics']['kv_live_blocks'],0)
    c.ok('泄漏没有被错误记成输出质量下降',leak['stage_slo_bad_versions'].get('canary',0)==0)
    for r in (healthy,bad,global_report,leak):
        c.ok('报告无原始请求正文_'+r['scenario'],all('hello' not in canonical(x) for x in r['traces']))
        c.equal('所有场景真实KV页最终清零_'+r['scenario'],r['metrics']['kv_live_blocks'],0)
        c.ok('审计哈希链具有有效root_'+r['scenario'],len(r['audit_summary']['root_hash'])==64)
        c.ok('各请求只产生一个终态_'+r['scenario'],len({t['trace_id'] for t in r['telemetry']})==len(r['telemetry']))
    c.ok('SLO双窗口场景至少两阶段证据',all(len(r['decisions'])>=2 for r in (healthy,bad,global_report)))
    c.ok('KV页压力可在首阶段提前保护',len(leak['decisions'])>=1)

    # 在初始阶段调用双窗口评估，未满每侧配额不可仅凭第一个坏请求回滚。
    gate=IncrementalGateway(tok,models,cfg); ctl=IncidentController(gate,gate.baseline_hash)
    ctl.start(50)
    used=set()
    stage_traffic(gate,{'stable':6,'canary':6},used,'slow')
    scarce=assess_canary(gate,cfg)
    c.equal('单窗口充分但长窗口不足只等待',scarce['reason'],'insufficient_per_version_samples')
    c.equal('样本不足不能回滚',ctl.consider(scarce),'wait')
    stage_traffic(gate,{'stable':6,'canary':6},used,'slow')
    decision=assess_canary(gate,cfg)
    c.equal('达到双窗口后产生roll_back',decision['action'],'rollback')
    c.close('灰度短窗口20倍burn',decision['evidence']['windows']['canary']['short']['burn'],20.)
    c.close('稳定短窗口0倍burn',decision['evidence']['windows']['stable']['short']['burn'],0.)
    # 旧canary统计已满足数量，但若随后大量只有stable请求，不能用陈旧窗口执行回滚。
    delayed=IncrementalGateway(tok,models,cfg);delayed_ctl=IncidentController(delayed,delayed.baseline_hash)
    delayed_ctl.start(50);delayed_used=set()
    stage_traffic(delayed,{'stable':12,'canary':12},delayed_used,'slow')
    c.equal('新鲜双窗口可识别灰度故障',assess_canary(delayed,cfg)['action'],'rollback')
    stage_traffic(delayed,{'stable':cfg.max_version_staleness+2},delayed_used,'none')
    overdue=assess_canary(delayed,cfg)
    c.equal('旧灰度窗口不能当近期事故证据',overdue['reason'],'stale_version_telemetry')
    c.equal('陈旧统计不触发回滚',delayed_ctl.consider(overdue),'wait')
    # 仅短窗口burn高、长期窗口仍低：不能省略双窗口条件。
    splitwindow=IncrementalGateway(tok,models,cfg)
    splitwindow.router.start_canary(50)
    splitused=set();stablekeys=find_keys(splitwindow.router,'stable',12,splitused)
    canarykeys=find_keys(splitwindow.router,'canary',12,splitused)
    for i in range(12):
        splitwindow.serve(ServiceRequest('orgA',stablekeys[i]))
        splitwindow.serve(ServiceRequest('orgA',canarykeys[i]),
                          fault=('slow' if i>=10 else 'none'))
    mixed=assess_canary(splitwindow,cfg)
    c.ok('只短窗超阈值但长窗未超阈值',
         mixed['evidence']['windows']['canary']['short']['burn']>=cfg.trigger_burn and
         mixed['evidence']['windows']['canary']['long']['burn']<cfg.trigger_burn)
    c.equal('单窗异常不能替代双窗触发',mixed['action'],'hold')
    stale=copy.deepcopy(decision);stale['evidence']['epoch']+=1
    c.rejects('陈旧epoch不能触发回滚',lambda:ctl.consider(stale))
    forged=copy.deepcopy(decision);forged['evidence']['windows']['canary']['short']['burn']=0
    c.rejects('篡改SLO证据不能触发回滚',lambda:ctl.consider(forged))
    c.equal('有真实证据才回滚',ctl.consider(decision),'rolled_back')
    c.equal('控制器回滚锁存',ctl.state,'stabilizing')
    c.ok('新请求回滚后只路由稳定版本',all(gate.router.choose(ServiceRequest('orgA',f'fresh{i}'))=='stable' for i in range(50)))
    c.rejects('回滚后未验证不得手动重新开启',lambda:ctl.manual_reenable(True))
    c.rejects('不足恢复样本拒绝',lambda:ctl.postcheck([]))
    c.equal('回滚之后无KV会话',gate.pool.usage()['live_blocks'],0)
    recovery=stage_traffic(gate,{'stable':cfg.recover_samples},used)
    c.rejects('虚构恢复结果不能蒙混过关',lambda:ctl.postcheck([copy.deepcopy(r) for r in recovery]))
    c.ok('真实稳定恢复通过',ctl.postcheck(recovery))
    c.equal('通过恢复检查以后已resolved',ctl.state,'resolved')
    c.rejects('人工批准不能跳过',lambda:ctl.manual_reenable(False,5))
    ctl.manual_reenable(True,5)
    c.equal('人工审批后进入监控',ctl.state,'monitoring')
    c.equal('二次灰度恢复到小权重',gate.router.weight,5)
    c.ok('重新灰度会改变epoch',gate.router.epoch>=3)
    c.ok('审计链审计数量大于3',gate.audit.verify()['count']>=3)
    chain=copy.deepcopy(gate.audit.rows)
    chain[1]['details']['reason']='forged'
    c.rejects('审计链篡改不能验证',lambda:verify_audit_rows(chain))
    gate.audit.rows[1]['details']['reason']='forged'
    c.rejects('实际审计篡改能发现',lambda:gate.audit.verify())

    # 回滚前已准入的旧版本必须保持固定；其完成与否是恢复验证的前置条件。
    pin=IncrementalGateway(tok,models,cfg);cp=IncidentController(pin,pin.baseline_hash)
    cp.start(50)
    key=find_keys(pin.router,'canary',1,set())[0]
    admitted,first=pin.admit(ServiceRequest('orgA',key))
    c.ok('灰度请求被准入但尚未执行',first and not admitted.completed)
    pin.router.rollback();cp.state='stabilizing'
    c.equal('路由关闭后已准入请求仍绑灰度',admitted.version,'canary')
    try:
        pin.router.set_weight(50)
        raise AssertionError('锁存失效')
    except LessonError:c.names.append('回滚锁存阻止新灰度')
    fresh=stage_traffic(pin,{'stable':cfg.recover_samples},{key})
    c.rejects('仍有未完成旧灰度时不能宣称恢复',lambda:cp.postcheck(fresh))
    complete,_=pin.execute(admitted,'slow')
    c.equal('旧准入完成仍使用原灰度',complete['version'],'canary')
    c.ok('旧灰度失败不会污染新epoch稳定验证',cp.postcheck(fresh))
    # 若回滚后的stable也持续退化，验证失败，不能靠路由权重为0宣布已恢复。
    failed_gateway=IncrementalGateway(tok,models,cfg)
    failed_controller=IncidentController(failed_gateway,failed_gateway.baseline_hash)
    failed_controller.start(50)
    failed_gateway.router.rollback();failed_controller.state='stabilizing'
    after_fault=stage_traffic(failed_gateway,{'stable':cfg.recover_samples},set(),'global_slow')
    c.ok('回滚后仍慢必须报告未恢复',failed_controller.postcheck(after_fault) is False)
    c.equal('未恢复状态需要人工处理',failed_controller.state,'needs_operator')
    c.rejects('未恢复不能人工重开灰度',lambda:failed_controller.manual_reenable(True,5))
    c.equal('路由固定未误切活跃版本',pin.pool.usage()['live_blocks'],0)

    # 实际KV容量故障：故意保留一份执行完成的canary KV占用。
    kv=IncrementalGateway(tok,models,DrillConfig(capacity_blocks=4,leak_pressure_fraction=.35))
    kv.router.start_canary(50)
    key=find_keys(kv.router,'canary',1,set())[0]
    row,_=kv.serve(ServiceRequest('orgA',key),fault='kv_leak')
    c.ok('注入泄漏确实占用物理页',kv.pool.usage()['live_blocks']>0)
    # 容量不足是明确失败，不是成功、EOS或隐式缓存替换。
    stablekey=find_keys(kv.router,'stable',1,{key})[0]
    badrow,_=kv.serve(ServiceRequest('orgA',stablekey))
    c.equal('真实页池不足以容量拒绝报告',badrow['terminal_status'],'capacity_rejected')
    c.ok('容量拒绝计入服务SLO失败',not badrow['slo_good'])
    kv.release_leaks()
    c.equal('释放泄漏后物理页回零',kv.pool.usage()['live_blocks'],0)
    later,_=kv.serve(ServiceRequest('orgA',find_keys(kv.router,'stable',1,{key,stablekey})[0]))
    c.ok('释放后稳定服务恢复',later['slo_good'])

    with tempfile.TemporaryDirectory() as tmp:
        save_models(tmp,tok,models)
        loaded_tok,loaded=load_models(tmp)
        c.equal('磁盘恢复Tokenizer身份',loaded_tok.contract(),tok.contract())
        for v in models:
            c.equal('独立模型重载哈希一致_'+v,state_hash(loaded[v].state_dict()),state_hash(models[v].state_dict()))
            c.ok('重载模型仍然冻结_'+v,all(not p.requires_grad for p in loaded[v].parameters()))
        bad_file=Path(tmp)/'canary.pt'
        obj=torch.load(bad_file,weights_only=True)
        first_adapter=next(iter(obj['state']))
        obj['state'][first_adapter].view(-1)[0]+=1
        torch.save(obj,bad_file)
        c.rejects('adapter内容篡改无法装载',lambda:load_models(tmp))
    return {'passed':len(c.names),'names':c.names,
            'scope':'确定性逻辑时间、真实CPU KV张量与两版本随机微型模型；非生产网络/SLO或GPU基准'}


def verify_audit_rows(rows):
    a=SafeAudit();a.rows=rows
    return a.verify()

In [25]:
checks=run_checks()
print('固定检查通过:',checks['passed'])
if RUN_DIR is not None:
    write_json(RUN_DIR/'checks.json',checks)

固定检查通过: 162


## 14. 独立练习：默认关闭，不由参考答案帮你填空

三道练习均允许先运行讲义其他部分；完成后再把对应开关改为 `True`。参考实现放在练习后，用独立函数名。

**练习 A：请求窗口与燃烧率**：输入终态列表、窗口长度 `n`、SLO目标，输出 `available/count/bad/burn`，样本不足时明确 `available=False`；不要把不足当作0失败。

尝试手算：10条中2条坏，目标95%，burn是多少倍？

In [26]:
def check_exercise_a(fn):
    c=Checks()
    rows=[{'slo_good':True},{'slo_good':False},{'slo_good':True},{'slo_good':False}]
    got=fn(rows,4,.95)
    c.equal('A_分母',got['count'],4)
    c.equal('A_bad',got['bad'],2)
    c.close('A_burn',got['burn'],10.)
    insufficient=fn(rows[:2],4,.95)
    c.ok('A_不足不能当作通过',not insufficient['available'])
    c.rejects('A_非法target拒绝',lambda:fn(rows,4,1.))
    return {'passed':len(c.names),'names':c.names}


def check_exercise_b(fn):
    c=Checks()
    healthy={'short':{'available':True,'bad_rate':0.,'burn':0.},
             'long':{'available':True,'bad_rate':0.,'burn':0.}}
    degraded={'short':{'available':True,'bad_rate':1.,'burn':20.},
              'long':{'available':True,'bad_rate':1.,'burn':20.}}
    early={'short':{'available':True,'bad_rate':1.,'burn':20.},
           'long':{'available':False}}
    c.equal('B_样本不足',fn(healthy,early),'wait')
    c.equal('B_灰度异常且稳定健康',fn(healthy,degraded),'rollback')
    c.equal('B_双版本异常人工处理',fn(degraded,degraded),'page')
    c.equal('B_都健康',fn(healthy,healthy),'hold')
    c.equal('B_仅短窗口异常不能回滚',fn(healthy,{
        'short':degraded['short'],'long':healthy['long']}),'hold')
    return {'passed':len(c.names),'names':c.names}


def check_exercise_c(fn):
    c=Checks();a=SafeAudit()
    a.append('canary_start',{'epoch':1,'weight':50})
    a.append('rollback',{'epoch':2,'canary_weight':0})
    original=copy.deepcopy(a.rows)
    root=fn(original)
    c.equal('C_root',root,a.verify()['root_hash'])
    tampered=copy.deepcopy(original);tampered[0]['details']['weight']=100
    c.rejects('C_篡改内容必须检出',lambda:fn(tampered))
    missing=copy.deepcopy(original);missing.pop(0)
    c.rejects('C_删除事件必须检出',lambda:fn(missing))
    reordered=copy.deepcopy(original);reordered.reverse()
    c.rejects('C_交换顺序必须检出',lambda:fn(reordered))
    return {'passed':len(c.names),'names':c.names}


def answer_a(rows,n,target):
    require(type(n) is int and n>0 and 0<target<1,'SLO窗口非法')
    chosen=rows[-n:]
    if len(chosen)<n:return {'available':False,'count':len(chosen),'required':n}
    bad=sum(not row['slo_good'] for row in chosen)
    return {'available':True,'count':n,'bad':bad,'burn':(bad/n)/(1-target)}


def answer_b(stable,canary):
    if not all(stable[w]['available'] and canary[w]['available'] for w in ('short','long')):
        return 'wait'
    if stable['short']['burn']>=4 and stable['long']['burn']>=4:
        return 'page'
    if (canary['short']['burn']>=4 and canary['long']['burn']>=4 and
        canary['short']['bad_rate']-stable['short']['bad_rate']>=.25 and
        stable['short']['bad_rate']<=.2):
        return 'rollback'
    return 'hold'


def answer_c(rows):
    previous='0'*64
    for idx,row in enumerate(rows):
        require(row['seq']==idx and row['prev']==previous,'审计序号/前驱不匹配')
        expected=digest({k:v for k,v in row.items() if k!='hash'})
        require(row['hash']==expected,'审计内容摘要不匹配')
        previous=row['hash']
    return previous

In [27]:
def my_window_sli(rows, n, target):
    raise NotImplementedError('请独立实现练习A：窗口与burn')

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:
    print(check_exercise_a(my_window_sli))

### 练习 B：双窗口归因

实现一个纯函数，输入stable和canary各自的short/long窗口结果，返回 `wait`、`rollback`、`page` 或 `hold`。

**优先级：**任何必需窗口不足先等待；稳定版本的两个窗口都严重退化时转人工；仅灰度的短长窗口高burn、稳定健康且差异显著时回滚；其他情况持有。

本题先用固定的4倍burn、0.25差异、0.2稳定版本坏率作为演练规则，不宣称它们是通用生产阈值。

In [28]:
def my_dual_window_gate(stable, canary):
    raise NotImplementedError('请独立实现练习B：wait/rollback/page/hold')

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:
    print(check_exercise_b(my_dual_window_gate))

### 练习 C：验证审计哈希链

实现检查每条审计事件的 `seq`、`prev`、内容摘要是否一致，正确则返回最终 root 哈希。删除事件、篡改细节或改动顺序必须抛出异常。

提醒：哈希链没有密钥和外部锚点，不能防御恶意攻击者同时重写所有记录；本题检查的是教学完整性。

In [29]:
def my_verify_audit(rows):
    raise NotImplementedError('请独立实现练习C：审计哈希链')

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:
    print(check_exercise_c(my_verify_audit))

## 15. 完整参考答案与独立练习验证

先完成自己的函数，再检查 `answer_a`、`answer_b`、`answer_c`。下面并不覆盖你的代码。固定检查和练习检查验证不同层次的目标：前者检查工程全部状态，后者检查你亲手实现的算法入口。

In [30]:
print('练习A参考通过:',check_exercise_a(answer_a))
print('练习B参考通过:',check_exercise_b(answer_b))
print('练习C参考通过:',check_exercise_c(answer_c))

练习A参考通过: {'passed': 5, 'names': ['A_分母', 'A_bad', 'A_burn', 'A_不足不能当作通过', 'A_非法target拒绝']}
练习B参考通过: {'passed': 5, 'names': ['B_样本不足', 'B_灰度异常且稳定健康', 'B_双版本异常人工处理', 'B_都健康', 'B_仅短窗口异常不能回滚']}
练习C参考通过: {'passed': 4, 'names': ['C_root', 'C_篡改内容必须检出', 'C_删除事件必须检出', 'C_交换顺序必须检出']}


## 16. 值班复盘：把事故动作与根因猜测分开

对本课四个演练按“观察 → 证据 → 控制动作 → 资源验证 → 留存边界”复盘：

| 观察 | 有证据的动作 | 不应该声称 |
|---|---|---|
| 健康灰度 | 继续监控 | “自动发布一定安全” |
| 只有灰度违反双窗口 | 锁存回滚，验证新稳定流量 | “回滚API成功 = 事故结束” |
| stable、canary同时严重变慢 | 升级人工处理、查共享原因 | “一定是新adapter导致” |
| 灰度KV页泄漏增压 | 隔离新的灰度流量、释放真实页 | “监控到压力=物理页已经释放” |

生产值班时还要定义事故指挥、沟通、责任人、时间线、客户影响和后续预防措施。本课只有离线技术演练，不做真实事故审批和发布。

## 17. CLI 入口和下一课

完整同名脚本还支持：

```bash
python lesson_79_serving_incident_slo_rollback.py --checks-only
python lesson_79_serving_incident_slo_rollback.py --math-only
python lesson_79_serving_incident_slo_rollback.py --output-dir lesson_79_outputs
python lesson_79_serving_incident_slo_rollback.py --reload-run lesson_79_outputs/实际目录
```

参考资料：

- [Google SRE：Alerting on SLOs](https://sre.google/workbook/alerting-on-slos/) — 错误预算、多时间窗口与低流量限制；本课是请求数窗口，不照搬阈值。
- [Google SRE：Managing Incidents](https://sre.google/sre-book/managing-incidents/) — 事故指挥、操作和沟通职责。
- [AWS CodeDeploy：Automatic Rollbacks](https://docs.aws.amazon.com/codedeploy/latest/userguide/deployments-rollback-and-redeploy.html) — 回滚有独立身份和状态。
- [OpenTelemetry Context Propagation](https://opentelemetry.io/docs/concepts/context-propagation/) — Trace 跨服务相关性；本课不是SDK实现。
- [PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html) — CPU环境复现也依赖版本和精度。

**下一课第 80 课建议：推理服务发布门禁、Runbook 与事故复盘自动化。** 将灰度证据、人工审批、验证清单和发布记录固化成可审计的离线流程。

In [31]:
def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    group=parser.add_mutually_exclusive_group()
    group.add_argument('--checks-only',action='store_true')
    group.add_argument('--reload-run',type=Path)
    group.add_argument('--math-only',action='store_true')
    parser.add_argument('--output-dir',type=Path,default=Path('lesson_79_outputs'))
    parser.add_argument('--seed',type=int,default=7901)
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.reload_run:
        print(json.dumps(reload_run(args.reload_run),ensure_ascii=False,indent=2));return 0
    if args.math_only:
        print(json.dumps({'burn_bad_2_of_10':window_sli([{'slo_good':i>=2} for i in range(10)],10,.95),
            'baseline':DrillConfig().__dict__},ensure_ascii=False,indent=2))
        return 0
    report=run_checks()
    print('固定检查通过：',report['passed'],'项。真实增量KV；时间是模拟逻辑tick。')
    if args.checks_only:return 0
    path=run_experiment(args.output_dir,args.seed)
    write_json(path/'checks.json',report)
    return 0